In [ ]:
!pip install transformers accelerate sentencepiece

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')
BASE_PATH = "/content/drive/MyDrive/Tubes/Summarization/"

CHECKPOINT_PATHS = {
    "BART-FULL": os.path.join(BASE_PATH, "experiment_full/bart_finetuned_summarizer"),
    "BART-SAMPLED": os.path.join(BASE_PATH, "experiment_sampled/bart_finetuned_summarizer"),
    "PEGASUS-FULL": os.path.join(BASE_PATH, "experiment_full/pegasus_finetuned_summarizer"),
    "PEGASUS-SAMPLED": os.path.join(BASE_PATH, "experiment_sampled/pegasus_finetuned_summarizer")
}

for name, path in CHECKPOINT_PATHS.items():
    if os.path.exists(path):
        print(f"Found {name} at {path}")
    else:
        print(f"Missing {name} at {path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found BART-FULL at /content/drive/MyDrive/Tubes/Summarization/experiment_full/bart_finetuned_summarizer
Found BART-SAMPLED at /content/drive/MyDrive/Tubes/Summarization/experiment_sampled/bart_finetuned_summarizer
Found PEGASUS-FULL at /content/drive/MyDrive/Tubes/Summarization/experiment_full/pegasus_finetuned_summarizer
Found PEGASUS-SAMPLED at /content/drive/MyDrive/Tubes/Summarization/experiment_sampled/pegasus_finetuned_summarizer


In [ ]:
import torch
from transformers import BartTokenizer, BartForConditionalGeneration, PegasusTokenizer, PegasusForConditionalGeneration
import gc

def load_model_and_tokenizer(model_key, model_path):
    """
    Loads the specific model and tokenizer based on the key (BART or PEGASUS).
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading {model_key} from {model_path} to {device}...")

    try:
        if "BART" in model_key:
            tokenizer = BartTokenizer.from_pretrained(model_path)
            model = BartForConditionalGeneration.from_pretrained(model_path).to(device)
        elif "PEGASUS" in model_key:
            tokenizer = PegasusTokenizer.from_pretrained(model_path)
            model = PegasusForConditionalGeneration.from_pretrained(model_path).to(device)
        else:
            raise ValueError("Model key must contain 'BART' or 'PEGASUS'")

        return tokenizer, model, device
    except Exception as e:
        print(f"Error loading {model_key}: {e}")
        return None, None, None

def generate_summary(text, tokenizer, model, device):
    """
    Generates a summary for a given text using standard generation parameters.
    """
    model.eval()

    # Preprocess input
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    ).to(device)

    # Generate summary
    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=128,
            min_length=10,
            length_penalty=2.0,
            num_beams=4,
            early_stopping=True
        )

    # Decode output
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

def clean_memory(model, tokenizer):
    """Deletes model and clears GPU cache to prevent OOM errors."""
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
input_text = """
Spacious parking lot, motorcycles, helmets, and cars are parked SAFELY.
It's better to come around 5 or 6 PM because it's easier to pick a good camping spot for photos.
The city light view at night is cool, and the view in the morning is no less cool.
The facilities are quite complete, there are toilets and prayer rooms.
Access to the location is quite easy, although the road is a bit winding.
"""

print(f"Original Text ({len(input_text.split())} words):\n{input_text}")

Original Text (73 words):

Spacious parking lot, motorcycles, helmets, and cars are parked SAFELY. 
It's better to come around 5 or 6 PM because it's easier to pick a good camping spot for photos. 
The city light view at night is cool, and the view in the morning is no less cool. 
The facilities are quite complete, there are toilets and prayer rooms. 
Access to the location is quite easy, although the road is a bit winding.



In [18]:
model_key = "BART-FULL"
if os.path.exists(CHECKPOINT_PATHS[model_key]):
    tokenizer, model, device = load_model_and_tokenizer(model_key, CHECKPOINT_PATHS[model_key])

    if model:
        summary = generate_summary(input_text, tokenizer, model, device)
        print(f"\n[{model_key}] Summary:\n{summary}\n")

        # Clean up to save memory for the next model
        clean_memory(model, tokenizer)
else:
    print(f"Skipping {model_key}: Checkpoint not found.")

Loading BART-FULL from /content/drive/MyDrive/Tubes/Summarization/experiment_full/bart_finetuned_summarizer to cpu...

[BART-FULL] Summary:
The attraction offers ample parking and is accessible by motorcycle, helmets, and cars. Visiting around 5 or 6 PM is recommended for optimal photo opportunities. Facilities include city light views at night and morning, along with toilets and prayer rooms. Road access is somewhat winding.



In [19]:
model_key = "BART-SAMPLED"
if os.path.exists(CHECKPOINT_PATHS[model_key]):
    tokenizer, model, device = load_model_and_tokenizer(model_key, CHECKPOINT_PATHS[model_key])

    if model:
        summary = generate_summary(input_text, tokenizer, model, device)
        print(f"\n[{model_key}] Summary:\n{summary}\n")

        # Clean up
        clean_memory(model, tokenizer)
else:
    print(f"Skipping {model_key}: Checkpoint not found.")

Loading BART-SAMPLED from /content/drive/MyDrive/Tubes/Summarization/experiment_sampled/bart_finetuned_summarizer to cpu...

[BART-SAMPLED] Summary:
This campsite offers ample parking and ample space for motorcycles, helmets, and cars. Visiting around 5 or 6 PM is recommended for easier photo opportunities. Facilities include toilets and prayer rooms, and the road is somewhat winding.



In [20]:
model_key = "PEGASUS-FULL"
if os.path.exists(CHECKPOINT_PATHS[model_key]):
    tokenizer, model, device = load_model_and_tokenizer(model_key, CHECKPOINT_PATHS[model_key])

    if model:
        summary = generate_summary(input_text, tokenizer, model, device)
        print(f"\n[{model_key}] Summary:\n{summary}\n")

        # Clean up
        clean_memory(model, tokenizer)
else:
    print(f"Skipping {model_key}: Checkpoint not found.")

Loading PEGASUS-FULL from /content/drive/MyDrive/Tubes/Summarization/experiment_full/pegasus_finetuned_summarizer to cpu...

[PEGASUS-FULL] Summary:
This attraction offers a spacious parking lot with motorcycles, helmets, and cars, making it ideal for photo opportunities. The site offers a cool city view at night and morning, with cool morning views. Facilities include toilets and prayer rooms.



In [21]:
model_key = "PEGASUS-SAMPLED"
if os.path.exists(CHECKPOINT_PATHS[model_key]):
    tokenizer, model, device = load_model_and_tokenizer(model_key, CHECKPOINT_PATHS[model_key])

    if model:
        summary = generate_summary(input_text, tokenizer, model, device)
        print(f"\n[{model_key}] Summary:\n{summary}\n")

        # Clean up
        clean_memory(model, tokenizer)
else:
    print(f"Skipping {model_key}: Checkpoint not found.")

Loading PEGASUS-SAMPLED from /content/drive/MyDrive/Tubes/Summarization/experiment_sampled/pegasus_finetuned_summarizer to cpu...

[PEGASUS-SAMPLED] Summary:
This attraction offers spacious parking, motorcycles, helmets, and cars, with easy access via a winding road. Visitors can enjoy cool city light views at night and cool morning views, making it ideal for photography. Facilities include toilets and prayer rooms.

